# Table of Contents

1. [Project Overview](#Project-Overview)
2. [Dataset](#dataset)
3. [Model Architecture (From Scratch)](#Model-Architecture-From-Scratch)
4. [Baseline Implementation](#Baseline-Implementation)
5. [Optimization Techniques](#Optimization-Techniques)
6. [Evaluation Metrics](#Evaluation-Metrics)
7. [Ablation Study](#Ablation-Study)
    - [Learning Rate Comparisons](#Learning-Rate-Comparisons)
    - [Regularization](#Regularization)
        - [Effect of Dropout Probability](#Effect-of-Dropout-Probability)
        - [Effect of L2 Regularization Strength](#Effect-of-L2-Regularization-Strength)
    - [Effect of Optimization Algorithm](#Effect-of-Optimization-Algorithm)
    - [Effect of Adam Hyperparameters](#Effect-of-Adam-Hyperparameters)
8. [Final Model Configuration](#Final-Model-Configuration)
9. [Results and Discussion](#Results-and-Discussion)
10. [Conclusion](#Conclusion)

## Project Overview

This project implements a neural network from scratch to classify CIFAR-10 images.  
The objective is to achieve at least 75% test accuracy.  

An ablation study is conducted to evaluate:
- Learning rate decay strategies  
- Regularization methods (L2, Dropout)  
- Optimization algorithms (SGD, Momentum, Adam)  
- Adam hyperparameter sensitivity  

## Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from utils.util import get_device
from evaluations import train_model, plot_metrics
from model.model import Model
import os

In [ ]:
# CIFAR-10 normalization stats
cifar_mean = [0.485, 0.456, 0.406]
cifar_std  = [0.229, 0.224, 0.225]


# Training transforms (with augmentation + normalization)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)
])


# Test transforms (NO augmentation, but WITH normalization)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)
])


training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)

#dynamic worker allocation
workers = os.cpu_count()
print(f"Setting number of workers to available CPU cores: {workers}")

from torch.utils.data import DataLoader
batch_size = 128
train_loader = torch.utils.data.DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True, persistent_workers=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True, persistent_workers=True)

### Data Format & Batch Creation

In [ ]:
image, label = training_data[0]
print("Image Shape: ", image.shape)
print("Label: ", label)

print("All Class Labels: \n", training_data.classes)

for images, labels in train_loader:
    print("images: [batch_size, channels, height, width]")
    print(images.shape)
    print("labels: [batch_size]")
    print(labels.shape)

    break


# Example images

In [ ]:
from utils.util import show_random_images
show_random_images(training_data)


## Create model (Pytorch Implementation)

### Layer Creation

In [ ]:
device = get_device()

# Abalation Study

In [ ]:
all_models = {}

def add_model(name, train_accs, test_accs, train_costs):
  all_models[name] = {
    "train_accs": train_accs,
    "test_accs": test_accs,
    "train_costs": train_costs
  }

## Base Model

In [ ]:
base_model, base_train_accs, base_test_accs, base_train_costs = train_model(
    model=Model(custom_loss="True"),
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=75,
    lr=0.01,
    optimizer_type="csi5140_gd",
    momentum=0,
    weight_decay=0.0,
    learn_rate_type=None,
    loss_funct="csi5140_loss"
)
add_model("SGD Base", base_train_accs, base_test_accs, base_train_costs)

plot_metrics(
    train_accs=base_train_accs,
    test_accs=base_test_accs,
    costs=base_train_costs,
    title_prefix="SGD Base Model"
)

## Regularization

### L2 Regularization 

In [ ]:
test_lambdas = [0, 1e-5, 1e-4, 5e-4, 1e-3]
models = {}
for i in range(len(test_lambdas)):
  lam = test_lambdas[i]
  model, train_accs, test_accs, train_costs = train_model(
      model=Model(custom_loss=True),
      train_loader=train_loader,
      test_loader=test_loader,
      device=device,
      epochs=75,
      lr=0.01,
      optimizer_type="csi5140_gd",
      weight_decay=lam,
      loss_funct="csi5140_loss"
  )
  models[lam] = {
      "model": model,
      "train_accs": train_accs,
      "test_accs": test_accs,
      "train_costs": train_costs
  }
  add_model(f"L2 λ={lam}", train_accs, test_accs, train_costs)
plt.figure()

for lam, data in models.items():
    epochs = range(1, len(data["train_accs"]) + 1)
    plt.plot(epochs, data["test_accs"], label=f"λ={lam} (test)")
    plt.plot(epochs, data["train_accs"], linestyle='--', label=f"λ={lam} (train)")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Weight Decay Comparison")
plt.legend()
plt.grid()
plt.show()

### Dropout (P = .3, .5, .7)

In [ ]:
test_p = [0, .3, .5, .7]
models = {}
for i in range(len(test_p)):
    p = test_p[i]
    model, train_accs, test_accs, train_costs = train_model(
        model=Model(dropout_p=p, custom_loss=True),
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=75,
        lr=0.01,
        optimizer_type="csi5140_gd",
        momentum=0,
        weight_decay=0,
        learn_rate_type=None,
        loss_funct="csi5140_loss"
    )
    
    models[p] = {
      "model": model,
      "train_accs": train_accs,
      "test_accs": test_accs,
      "train_costs": train_costs
    }
    add_model(f"Dropout p={p}", train_accs, test_accs, train_costs)
plt.figure()

for p, data in models.items():
    epochs = range(1, len(data["train_accs"]) + 1)
    plt.plot(epochs, data["test_accs"], label=f"p={p} (test)")
    plt.plot(epochs, data["train_accs"], linestyle='--', label=f"p={p} (train)")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Dropout Comparison")
plt.legend()
plt.grid()
plt.show()

## Optimization Algorithms

### Step Decay

In [ ]:
model, train_accs, test_accs, train_costs = train_model(
  model=Model(custom_loss=True),
  train_loader=train_loader,
  test_loader=test_loader,
  device=device,
  epochs=75,
  lr_max=0.01,
  step_size=5,
  gamma=0.5,
  optimizer_type="csi5140_gd",
  learn_rate_type="csi5140_step",
  weight_decay=0,
  loss_funct="csi5140_loss"
)
plot_metrics(
    train_accs=train_accs,
    test_accs=test_accs,
    costs=train_costs,
    title_prefix="Step Decay LR Model"
)
add_model(f"Step Decay", train_accs, test_accs, train_costs)

### Cosine Decay

In [ ]:
test_betas = [(0.9, 0.999), (0.8, 0.99), (0.7, 0.9)]

models = {}

for beta in test_betas:
    model, train_accs, test_accs, train_costs = train_model(
        model=Model(custom_loss=True),
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=75,
        #lr=0.01,
        optimizer_type="csi5140_gd",
        betas=beta,
        weight_decay=0,
        learn_rate_type = "csi5140_cosine",
        lr_min = 0.001, 
        lr_max = 0.01,
        loss_funct="csi5140_loss"
    )

    models[beta] = {
        "model": model,
        "train_accs": train_accs,
        "test_accs": test_accs,
        "train_costs": train_costs
    }
    add_model(f"SGD + Cosine β1={beta[0]}, β2={beta[1]}", train_accs, test_accs, train_costs)

In [ ]:

# --- Accuracy Plot ---
plt.figure()

for beta, data in models.items():
    epochs = range(1, len(data["test_accs"]) + 1)
    plt.plot(epochs, data["test_accs"], marker='o', label=f"{beta}")

plt.xlabel("Epoch")
plt.ylabel("Test Accuracy")
plt.title("Cosine Decay Test Accuracy vs Epoch (Betas)")
plt.legend()
plt.grid()
plt.show()


# --- Loss Plot ---
plt.figure()

for beta, data in models.items():
    # If stored as (iteration, loss)
    add_model(f"Cosine β={beta}", data["train_accs"], data["test_accs"], data["train_costs"])
    if isinstance(data["train_costs"][0], tuple):
        iterations, losses = zip(*data["train_costs"])
        plt.plot(iterations, losses, label=f"{beta}")
    else:
        # If it's just a list of losses per step
        plt.plot(data["train_costs"], label=f"{beta}")

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Training Loss vs Iteration (Betas)")
plt.legend()
plt.grid()
plt.show()

### Gradient Descent With Momentum

In [ ]:
model, train_accs, test_accs, train_costs = train_model(
  model=Model(custom_loss=True),
  train_loader=train_loader,
  test_loader=test_loader,
  device=device,
  epochs=75,
  lr=0.01,
  optimizer_type="csi5140_gdm",
  momentum=0.5,
  weight_decay=0,
  loss_funct="csi5140_loss"
)
add_model("SGD + Momentum (0.9)", train_accs, test_accs, train_costs)
plot_metrics(
    train_accs=train_accs,
    test_accs=test_accs,
    costs=train_costs,
    title_prefix="SGD With Momentum Model"
)

### Adam - Beta1 & Beta2

In [ ]:
test_betas = [(0.9, 0.999), (0.8, 0.99), (0.7, 0.9)]

models = {}

for beta in test_betas:
  model, train_accs, test_accs, train_costs = train_model(
    model=Model(custom_loss=True),
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=75,
    lr=0.001,
    optimizer_type="csi5140_adam",
    betas=beta,
    weight_decay=0,
    loss_funct="csi5140_loss"
  )

  models[beta] = {
    "model": model,
    "train_accs": train_accs,
    "test_accs": test_accs,
    "train_costs": train_costs
  }
  add_model(f"Adam β1={beta[0]}, β2={beta[1]}", train_accs, test_accs, train_costs)

### (Plots) Adam - Beta1 & Beta2

In [ ]:
# get default color 
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# --- Accuracy Plot ---
plt.figure()

for i, (beta, data) in enumerate(models.items()):
    epochs = range(1, len(data["train_accs"]) + 1)
    color = colors[i % len(colors)]
    
    # train = lighter 
    plt.plot(epochs, data["train_accs"],
        linestyle='--', color=color, alpha=0.5,
        label=f"{beta} (train)")
    
    # test = solid + full color
    plt.plot(epochs, data["test_accs"],
        linestyle='-', color=color, alpha=1.0,
        label=f"{beta} (test)")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Test Accuracy for Different Betas")
plt.legend()
plt.grid()
plt.show()


# --- Loss Plot ---
plt.figure()

for i, (beta, data) in enumerate(models.items()):
    iterations, losses = zip(*data["train_costs"])
    color = colors[i % len(colors)]
    
    plt.plot(iterations, losses, color=color, label=f"{beta}")

plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss vs Iterations for Different Betas")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# --- Main Model ---
main_model, main_train_accs, main_test_accs, main_train_costs = train_model(
    model=Model(custom_loss=True),
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=75,
    lr_min=0.0001,
    betas = (0.9, 0.999),
    lr_max=0.001,
    optimizer_type="csi5140_adam",
    weight_decay=1e-4,
    learn_rate_type="csi5140_cosine",
    loss_funct="csi5140_loss"
)
add_model("Adam Final", main_train_accs, main_test_accs, main_train_costs)

# Plot main model
plot_metrics(
    train_accs=main_train_accs,
    test_accs=main_test_accs,
    costs=main_train_costs,
    title_prefix="Final Adam + Cosine Model"
)

In [ ]:
# --- Select BEST models from each experiment ---

best_models = {}

def get_best_model(prefix):
    candidates = {k: v for k, v in all_models.items() if prefix in k}
    if not candidates:
        return None
    return max(candidates.items(), key=lambda x: max(x[1]["test_accs"]))

# Pick best from each category
categories = [
    "SGD Base",
    "L2",
    "Dropout",
    "SGD + Cosine",
    "Adam β",
    "SGD + Momentum",
    "Exponential",
    "Adam Final"
]

for cat in categories:
    result = get_best_model(cat)
    if result:
        name, data = result
        best_models[name] = data

print("Selected Models:")
for name in best_models:
    print(name)
plt.figure()

for name, data in best_models.items():
    epochs = range(1, len(data["test_accs"]) + 1)
    plt.plot(epochs, data["test_accs"], marker='o', label=name)

plt.xlabel("Epoch")
plt.ylabel("Test Accuracy")
plt.title("Best Model from Each Experiment")
plt.legend(fontsize=8)
plt.grid()
plt.show()